In [19]:
import numpy as np
import CoolProp.CoolProp as CP
from scipy.integrate import quad

# =============================================================================
# 1. INPUT PARAMETERS (SI Units & Conversions matching EES)
# =============================================================================
m_inlet_water = 0.1             # Rate of incoming water [kg/s]
T_inlet_water = 40.0 + 273.15   # 40 °C converted to Kelvin [K]
P_inlet_water = 100 * 1000      # 100 kPa converted to Pascal [Pa]
P_outlet_H2   = 100 * 1000      # 100 kPa converted to Pascal [Pa]

lambda_a = 14.0                 # Water content at the anode-membrane
lambda_c = 10.0                 # Water content at the cathode-membrane
D        = 0.0001               # Membrane thickness [m]

J_ref_a = 170000.0              # Anode pre-exponential factor [A/m^2]
J_ref_c = 4600.0                # Cathode pre-exponential factor [A/m^2]
E_act_a = 76000.0               # Activation energy for the anode [J/mol]
E_act_c = 18000.0               # Activation energy for the cathode [J/mol]

F      = 96486.0                # Faraday constant [C/mol]
R      = 8.3145                 # Universal Gas Constant [J/mol-K]
LHV_H2 = 242.847 * 1000         # kJ/mol converted to J/mol

J = 5000.0                      # Current Density [A/m^2]

h_op = 2072.0                   # Hours of operation in a year
s_op = h_op * 3600.0            # Seconds of operation in a year

N_cells = 1                     # Number of cells in stack

print("--- Step 1: PEME Design Inputs Initialized Successfully ---")

--- Step 1: PEME Design Inputs Initialized Successfully ---


In [20]:
# =============================================================================
# 2. ENTHALPY AND ENTROPY CALCULATIONS (Via CoolProp)
# =============================================================================

# Inlet Water
H_inlet_water = CP.PropsSI('H', 'T', T_inlet_water, 'P', P_inlet_water, 'Water')
s_inlet_water = CP.PropsSI('S', 'T', T_inlet_water, 'P', P_inlet_water, 'Water')

# Outlet Oxygen
T_outlet_O2 = T_inlet_water
P_outlet_O2 = P_outlet_H2
H_outlet_O2 = CP.PropsSI('H', 'T', T_outlet_O2, 'P', P_outlet_O2, 'Oxygen')
s_outlet_O2 = CP.PropsSI('S', 'T', T_outlet_O2, 'P', P_outlet_O2, 'Oxygen')

# Outlet Hydrogen
T_outlet_H2 = T_inlet_water
H_outlet_H2 = CP.PropsSI('H', 'T', T_outlet_H2, 'P', P_outlet_H2, 'Hydrogen')
s_outlet_H2 = CP.PropsSI('S', 'T', T_outlet_H2, 'P', P_outlet_H2, 'Hydrogen')

print("--- Step 2: CoolProp Thermodynamic States Completed ---")

--- Step 2: CoolProp Thermodynamic States Completed ---


In [21]:
# =============================================================================
# 3. HYDROGEN PRODUCTION & ELECTROCHEMICAL LOSSES
# =============================================================================

# Faraday's Law Mass & Molar Splits
N_dot_H2_out      = N_cells * J / (2.0 * F)
N_dot_H2O_reacted = N_cells * N_dot_H2_out
N_dot_O2_out      = N_cells * J / (4.0 * F)
N_dot_H2O_out     = (m_inlet_water * 1000.0 / 18.0) - N_dot_H2O_reacted

# 1. Reversible Potential (Nernst Equation with Pressure Corrections)
V_0_base  = 1.229 - (8.5 * 0.0001 * (T_inlet_water - 298.15))
V_0_press = (R * T_inlet_water / F) * np.log(np.sqrt(P_outlet_H2 * P_outlet_O2) / P_inlet_water)
V_0       = V_0_base + V_0_press

# 2. Ohmic Overpotential via Numerical Integration across Thickness D
def dydx_func(x):
    lambda_x = (((lambda_a - lambda_c) / D) * x) + lambda_c
    sigma_lambda_x = (0.5139 * lambda_x - 0.326) * np.exp(1268.0 * ((1.0 / 303.0) - (1.0 / T_inlet_water)))
    return 1.0 / sigma_lambda_x

# Perform the exact 0 to D definite integration
R_ohm, _ = quad(dydx_func, 0, D)
V_ohm    = J * R_ohm

# 3. Activation Overpotential (Butler-Volmer via Inverse Hyperbolic Sine)
J_0_a = J_ref_a * np.exp(-E_act_a / (R * T_inlet_water))
J_0_c = J_ref_c * np.exp(-E_act_c / (R * T_inlet_water))

V_act_a = (R * T_inlet_water / F) * np.arcsinh(J / (2.0 * J_0_a))
V_act_c = (R * T_inlet_water / F) * np.arcsinh(J / (2.0 * J_0_c))

# 4. Total Operating System Voltage
V_cell  = V_0 + V_act_a + V_act_c + V_ohm
V_stack = V_cell * N_cells

print("--- Step 3: Polarisation Model Fully Resolved ---")

--- Step 3: Polarisation Model Fully Resolved ---


In [22]:
# =============================================================================
# 4. MASS BALANCE & PERFORMANCE CALCULATIONS
# =============================================================================

# Absolute mass flow rates [kg/s]
m_H2_out           = 2.0 * N_dot_H2_out * 0.001
m_dot_H2O_out      = 18.0 * N_dot_H2O_out * 0.001
m_O2_out           = 32.0 * N_dot_O2_out * 0.001
m_O2_H2O_out       = m_O2_out + m_dot_H2O_out
m_dot_H2O_reacted  = 18.0 * N_dot_H2O_reacted * 0.001

# System Power Input [kW] (Note: J * V_stack = Watts, divided by 1000 = kW)
W_dot_elec = (J * V_stack) / 1000.0
eta_th_H2  = (LHV_H2 * N_dot_H2_out) / (W_dot_elec * 1000.0)

# Annual production quantities [kg/year]
m_dot_H2_prod = m_H2_out * s_op
m_dot_O2_prod = m_O2_out * s_op

# =============================================================================
# 5. EXPANDED EXPLICIT PORTFOLIO PRINTER
# =============================================================================
print(f"==================================================")
print(f"      PEME SYSTEM CONVERGENCE          ")
print(f"==================================================")
print(f"Cell Operating Voltage (V_cell):   {V_cell:.3f} V  ")
print(f"Stack Operating Voltage (V_stack): {V_stack:.3f} V  ")
print(f"Total Power Input (W_dot_elec):    {W_dot_elec:.2f} kW ")
print(f"Thermal LHV Efficiency:            {eta_th_H2 * 100:.2f} % ")
print(f"--------------------------------------------------")
print(f"H2 Mass Flow Output Rate:          {m_H2_out:.8f} kg/s")
print(f"Annual Hydrogen Yield:             {m_dot_H2_prod:.2f} kg/year")
print(f"==================================================")

      PEME SYSTEM CONVERGENCE          
Cell Operating Voltage (V_cell):   2.173 V  
Stack Operating Voltage (V_stack): 2.173 V  
Total Power Input (W_dot_elec):    10.87 kW 
Thermal LHV Efficiency:            57.91 % 
--------------------------------------------------
H2 Mass Flow Output Rate:          0.00005182 kg/s
Annual Hydrogen Yield:             386.54 kg/year
